# tp-25: 走法特性スコアリング (TS / TS持続力 / GC)

各馬の個別ラップから3つのスコアを算出する。

- **TS (トップスピード)**: L3-1F における最高速度のレース内偏差値
- **TS持続力**: L4-1F で最高速度をどれだけ維持できたかの得点
- **GC (ギアチェンジ)**: L5-1F における追走ペースから最高速度への転換の鋭さ

**データの制約**: レース全体ラップ + 各馬の上がり3F のみ利用可能。
個別ラップ推定は「補正ラップ手法」: `horse_lap_i = race_lap_i × (horse_last3f / race_last3f_sum)`

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../../..').resolve()))

import ast
import numpy as np
import pandas as pd
from scipy.stats import zscore
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

ROOT = Path('../../..').resolve()
LAP_DIR = ROOT / 'data/local/backup/tables_before_reexport_20261014'
OUT_DIR = ROOT / 'data/local/modeling/training_pipeline/lapscores'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)

## 1. データ読み込み

In [ ]:
YEARS = ['2020', '2021', '2022', '2023', '2024', '2025']

dfs = []
for y in YEARS:
    p = LAP_DIR / y / 'race_result_lap_flat.parquet'
    if p.exists():
        d = pd.read_parquet(p)
        d['year'] = y
        dfs.append(d)
        print(f'{y}: {len(d):,} rows')
    else:
        print(f'{y}: NOT FOUND')

df = pd.concat(dfs, ignore_index=True)
print(f'\n合計: {len(df):,} rows, {df["race_id"].nunique():,} races')
print(df.dtypes[['lap_times','last_3f','distance']])

## 2. ラップタイムのパース

In [ ]:

import ast as _ast

# ラップタイムのF数カウント
df['n_laps'] = df['lap_times'].map(
    lambda s: len(_ast.literal_eval(s)) if isinstance(s, str) else (len(s) if s else 0)
)
df['last_3f'] = pd.to_numeric(df['last_3f'], errors='coerce')

# 有効行フィルタ: 5F以上 & last_3f有効
valid = (df['n_laps'] >= 5) & df['last_3f'].notna() & (df['last_3f'] > 0)
df = df[valid].copy().reset_index(drop=True)
print(f'有効行: {len(df):,} ({valid.mean():.1%})')
print('レース長別(F数):', df['n_laps'].value_counts().sort_index().to_dict())


## 3. 補正ラップによる個別推定

各馬の L3〜L1 推定時計 = `race_lap_i × (horse_last3f / race_last3f_sum)`

L4・L5 はレース全体ラップをそのまま使用（個別差は省略）。

In [ ]:

# ベクトル化: 一括ラップパース・展開
print('ラップ列を抽出中...')
laps_list = [
    _ast.literal_eval(s) if isinstance(s, str) else list(s)
    for s in df['lap_times']
]

df['race_last3f'] = [sum(l[-3:]) if len(l) >= 3 else float('nan') for l in laps_list]
df['lap_L1']  = [l[-1] if len(l) >= 1 else float('nan') for l in laps_list]
df['lap_L2']  = [l[-2] if len(l) >= 2 else float('nan') for l in laps_list]
df['lap_L3']  = [l[-3] if len(l) >= 3 else float('nan') for l in laps_list]
df['race_L4'] = [l[-4] if len(l) >= 4 else float('nan') for l in laps_list]
df['race_L5'] = [l[-5] if len(l) >= 5 else float('nan') for l in laps_list]

# 補正ラップ: 馬ごとのスケール係数
scale = df['last_3f'] / df['race_last3f']
df['est_L3'] = df['lap_L3'] * scale
df['est_L2'] = df['lap_L2'] * scale
df['est_L1'] = df['lap_L1'] * scale

print('推定ラップ完了')
print(df[['est_L1','est_L2','est_L3','race_L4','race_L5']].describe().round(2))


## 4. スコア計算

### 4-1. TS (トップスピード) スコア
- 馬のL3-L1の最速F = `min(est_L3, est_L2, est_L1)` (時間が短い=速い)
- スピード指数 = `12.0 / fastest_F` × 100 (12.0秒基準)
- レース内偏差値 (平均50, SD10)

In [ ]:

# TS: L3-L1の最速フロン
df['ts_fastest_f'] = df[['est_L3','est_L2','est_L1']].min(axis=1)
df['ts_idx'] = 12.0 / df['ts_fastest_f'] * 100   # スピード指数

# レース内偏差値: groupby.transform で高速化
mu = df.groupby('race_id')['ts_idx'].transform('mean')
sd = df.groupby('race_id')['ts_idx'].transform('std').fillna(1.0).replace(0.0, 1.0)
df['ts_score'] = (df['ts_idx'] - mu) / sd * 10 + 50

print('TS スコア分布:')
print(df['ts_score'].describe().round(2))


### 4-2. TS持続力スコア

**データ制約についての注記**:  
`last_3f` 比例スケーリングでは L3:L2:L1 の比率が全馬同一になるため、  
「持続力の馬間差」は TS (last_3f の絶対速度) にほぼ吸収される。

代替アプローチ: **過去走実績の平均TS/GC** を持続力指標とする。  
ここでは当初の定義も計算するが、主な価値は過去走集計特徴量として利用すること。


In [ ]:

# TS持続力: ベクトル演算
peak = df['ts_fastest_f']
thr  = peak * 1.05

L3, L2, L1 = df['est_L3'], df['est_L2'], df['est_L1']
L4 = df['race_L4']

sustained = (
    (L4 <= thr).astype(float) +
    (L3 <= thr).astype(float) +
    (L2 <= thr).astype(float) +
    (L1 <= thr).astype(float)
)
bonus = (
    0.5 * ((L3 > L2) & (L2 > L1)).astype(float) +   # 連続加速
    0.5 * (L4 <= thr).astype(float)                   # L4もピーク域
)
df['ts_stamina_raw'] = (sustained + bonus).clip(0, 5)
# L4 が NaN の行はそのまま NaN に
df.loc[L4.isna(), 'ts_stamina_raw'] = np.nan

# レース内偏差値
mu2 = df.groupby('race_id')['ts_stamina_raw'].transform('mean')
sd2 = df.groupby('race_id')['ts_stamina_raw'].transform('std').fillna(1.0).replace(0.0, 1.0)
df['ts_stamina_score'] = (df['ts_stamina_raw'] - mu2) / sd2 * 10 + 50

print('TS持続力スコア分布:')
print(df['ts_stamina_score'].describe().round(2))


### 4-3. GC (ギアチェンジ) スコア

L5-L1 で追走ペースから最高速への切り替えの鋭さを評価。

**アルゴリズム**:
1. 追走ペース = L5 (レース値)
2. ピーク = min(est_L3, est_L2, est_L1)
3. 加速幅 = (L5 - peak) / L5  (大きいほど大きな加速)
4. ピーク位置 = L3が最速なら pos=1, L2なら2, L1なら3 (早い位置=早い切り替え)
5. GC_raw = 加速幅 × (4 - peak_pos) / 3  (早く大きな加速 = 高スコア)
6. レース内偏差値化

In [ ]:

# GC: ベクトル演算
L5 = df['race_L5']
peak = df[['est_L3','est_L2','est_L1']].min(axis=1)
accel_mag = (L5 - peak) / L5.replace(0, np.nan)

# ピーク位置: argmin (0=L3最速, 1=L2最速, 2=L1最速)
fastest_idx = df[['est_L3','est_L2','est_L1']].values.argmin(axis=1)
peak_pos = fastest_idx + 1   # 1=L3, 2=L2, 3=L1

df['gc_raw'] = accel_mag.values * (4 - peak_pos) / 3.0

# レース内偏差値
mu3 = df.groupby('race_id')['gc_raw'].transform('mean')
sd3 = df.groupby('race_id')['gc_raw'].transform('std').fillna(1.0).replace(0.0, 1.0)
df['gc_score'] = (df['gc_raw'] - mu3) / sd3 * 10 + 50

print('GCスコア分布:')
print(df['gc_score'].describe().round(2))


## 5. スコアの検証

各スコアが実際のレース結果と相関しているかを確認。

In [ ]:

# layer_a から着順を取得してマージ
layer_a = pd.read_parquet(ROOT / 'data/local/modeling/training_pipeline/layer_a.parquet',
                          columns=['race_id','horse_id','finish_position'])

eval_df = df.merge(layer_a, on=['race_id','horse_id'], how='inner')
print(f'マージ後: {len(eval_df):,} rows ({eval_df["race_id"].nunique():,} races)')

# 着順との相関
from scipy.stats import spearmanr

scores = ['ts_score', 'ts_stamina_score', 'gc_score']
print('\nSpearman相関 vs 着順 (負 = スコア高い→着順良い):')
for s in scores:
    mask = eval_df[s].notna()
    r, p = spearmanr(eval_df.loc[mask, s], eval_df.loc[mask, 'finish_position'])
    print(f'  {s}: r={r:.4f}  p={p:.3e}')


In [ ]:

# スコア上位馬の勝率・複勝率を確認
print('=== スコアパーセンタイル別 勝率・複勝率 ===')

for score_col in scores:
    print(f'\n--- {score_col} ---')
    rows = []
    for pct in [50, 70, 80, 90]:
        thr = eval_df[score_col].quantile(pct/100)
        sel = eval_df[eval_df[score_col] >= thr]
        win_rate  = (sel['finish_position'] == 1).mean()
        top3_rate = (sel['finish_position'] <= 3).mean()
        rows.append({'上位%': f'上位{100-pct}%', '選択数': len(sel),
                     '勝率': f'{win_rate:.3f}', '複勝率': f'{top3_rate:.3f}'})
    print(pd.DataFrame(rows).to_string(index=False))

br_win  = (eval_df['finish_position'] == 1).mean()
br_top3 = (eval_df['finish_position'] <= 3).mean()
print(f'\nベースライン: 勝率={br_win:.3f}, 複勝率={br_top3:.3f}')


In [ ]:
# 可視化: スコア分布 + 着順との関係
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, s in enumerate(scores):
    # スコア分布
    ax = axes[0, i]
    eval_df[s].hist(bins=50, ax=ax, color='steelblue', edgecolor='none')
    ax.set_title(f'{s} 分布', fontsize=11)
    ax.set_xlabel('偏差値スコア')
    ax.axvline(50, color='red', linestyle='--', alpha=0.5)
    
    # 着順別スコア箱ひげ
    ax2 = axes[1, i]
    top5 = eval_df[eval_df['finish_position'] <= 5]
    others = eval_df[eval_df['finish_position'] > 5]
    ax2.boxplot(
        [top5[s].dropna(), others[s].dropna()],
        labels=['1-5着', '6着以下'],
        notch=True, patch_artist=True,
        boxprops=dict(facecolor='lightblue'),
    )
    ax2.set_title(f'{s} vs 着順', fontsize=11)
    ax2.set_ylabel('スコア')

plt.tight_layout()
plt.savefig(OUT_DIR / 'score_validation.png', dpi=120, bbox_inches='tight')
plt.show()
print('保存:', OUT_DIR / 'score_validation.png')

## 6. 馬ごとの過去実績集計 (特徴量化)

各スコアを「この馬の過去履歴の平均・最大」として集計し、  
将来レースの予測特徴量として使えるようにする。

In [ ]:
# レース全体スコアの整理
score_df = df[[
    'race_id', 'date', 'horse_id', 'horse_name', 'year',
    'distance', 'surface', 'venue',
    'ts_score', 'ts_stamina_score', 'gc_score',
    'ts_fastest_f', 'ts_stamina_raw', 'gc_raw',
    'est_L1', 'est_L2', 'est_L3', 'race_L4', 'race_L5',
]].copy()
score_df['date'] = pd.to_datetime(score_df['date'])
score_df = score_df.sort_values(['horse_id','date']).reset_index(drop=True)

print('スコアDF:', score_df.shape)
print(score_df[['ts_score','ts_stamina_score','gc_score']].describe().round(2))

In [ ]:

# 馬の過去N走の平均/最大/最新値 (time-leak防止: shift(1))
score_df = score_df.sort_values(['horse_id','date']).reset_index(drop=True)

score_cols = ['ts_score', 'ts_stamina_score', 'gc_score']
windows = [3, 5]

for s in score_cols:
    for w in windows:
        score_df[f'{s}_avg{w}'] = (
            score_df.groupby('horse_id')[s]
            .transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
        )
        score_df[f'{s}_max{w}'] = (
            score_df.groupby('horse_id')[s]
            .transform(lambda x: x.shift(1).rolling(w, min_periods=1).max())
        )
    score_df[f'{s}_last1'] = score_df.groupby('horse_id')[s].transform(lambda x: x.shift(1))

feat_cols = [c for c in score_df.columns if any(c.startswith(sc) for sc in score_cols) and ('avg' in c or 'max' in c or 'last' in c)]
print('生成した特徴量:', feat_cols)
print(score_df[feat_cols].describe().round(2))


## 7. 保存

In [ ]:

# race-horse レベルのスコアを保存
out_scores = score_df[[
    'race_id', 'horse_id', 'date', 'year',
    'ts_score', 'ts_stamina_score', 'gc_score',
    'ts_fastest_f', 'ts_stamina_raw', 'gc_raw',
] + feat_cols]

out_scores.to_parquet(OUT_DIR / 'lapscores.parquet', index=False)
print('保存:', OUT_DIR / 'lapscores.parquet')
print('shape:', out_scores.shape)

print('\n--- 有効な特徴量 (TS/GC系) ---')
print('ts_score_avg3, ts_score_max3, ts_score_avg5, ts_score_max5, ts_score_last1')
print('gc_score_avg3, gc_score_max3, gc_score_avg5, gc_score_max5, gc_score_last1')
print('\n--- 注意: ts_stamina系はレース内差異が小さいため過去集計でのみ有効 ---')


In [ ]:
# スコア別 top/bottom 馬の例 (直感的な検証)
print('=== TS スコア上位 10 馬 (2022-2024) ===')
top_ts = score_df[score_df['year'].isin(['2022','2023','2024'])].nlargest(10, 'ts_score')
print(top_ts[['horse_name','race_id','ts_score','ts_fastest_f','distance','venue']].to_string(index=False))

print('\n=== GC スコア上位 10 馬 (2022-2024) ===')
top_gc = score_df[score_df['year'].isin(['2022','2023','2024'])].nlargest(10, 'gc_score')
print(top_gc[['horse_name','race_id','gc_score','gc_raw','distance','venue']].to_string(index=False))

In [ ]:
# 有名馬の走法プロファイル確認 (イクイノックス等)
famous = ['イクイノックス', 'リバティアイランド', 'ドウデュース', 'タスティエーラ']
for name in famous:
    horse_rows = score_df[score_df['horse_name'] == name].tail(5)
    if len(horse_rows) == 0:
        continue
    print(f'\n--- {name} ---')
    print(horse_rows[['date','venue','distance','ts_score','ts_stamina_score','gc_score']].to_string(index=False))